# QET 3-qubit ground state: null space of $H_{\mathrm{tot}}$ and zero-energy conditions

This notebook demonstrates two facts about the entangled ground state $|g\rangle$ of the 3-qubit
QET Hamiltonian:

1. **Null space.** $|g\rangle$ is the *null space* (kernel) of the total Hamiltonian, i.e.
   $H_{\mathrm{tot}}\,|g\rangle = 0$. Equivalently, $|g\rangle$ is the unique eigenvector with
   eigenvalue $0$, and all other eigenvalues are strictly positive — so the null space is
   one-dimensional and spanned exactly by $|g\rangle$.

2. **Zero-mean energy for every part.** $|g\rangle$ satisfies
   $$\langle g|H_0|g\rangle=\langle g|H_1|g\rangle=\langle g|H_2|g\rangle
     =\langle g|V_{0,1}|g\rangle=\langle g|V_{1,2}|g\rangle=\langle g|V_{0,2}|g\rangle
     =\langle g|H_{\mathrm{tot}}|g\rangle=0,$$
   for every local field $H_n$, every interaction bond $V_{i,j}$, and the total Hamiltonian.

Conventions: qubits $q_0,q_1,q_2$; $H_n=hZ_n+c_H$, $V_{i,j}=kX_iX_j+c_V$, with the closed-form
shifts $c_H=h(1-M^2)/(1+3M^2)$, $c_V=2kM(1-M)/(1+3M^2)$ and $M=k/(2h+k+2K)$,
$K=\sqrt{h^2+hk+k^2}$.

## 0. Setup

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

I2 = np.eye(2)
X  = np.array([[0, 1], [1, 0]], float)
Z  = np.array([[1, 0], [0, -1]], float)

def op(q, M, n=3):
    # place 2x2 matrix M on qubit q (q0 = least significant)
    f = [I2] * n
    f[n - 1 - q] = M
    out = f[0]
    for t in f[1:]:
        out = np.kron(out, t)
    return out

def K_(h, k):  return np.sqrt(h*h + h*k + k*k)
def M_(h, k):  return k / (2*h + k + 2*K_(h, k))

def constants(h, k):
    M = M_(h, k)
    cH = h * (1 - M**2) / (1 + 3*M**2)
    cV = 2 * k * M * (1 - M) / (1 + 3*M**2)
    return cH, cV

def ground_state(h, k):
    M = M_(h, k); N = np.sqrt(1 + 3*M*M)
    g = np.zeros(8)
    g[[1, 2, 4]] = -M / N      # |001>, |010>, |100>
    g[7] = 1 / N               # |111>
    return g

def H_local(h, k, n):
    cH, _ = constants(h, k)
    return h * op(n, Z) + cH * np.eye(8)

def V_bond(h, k, i, j):
    _, cV = constants(h, k)
    return k * op(i, X) @ op(j, X) + cV * np.eye(8)

def H_tot(h, k):
    H = sum(H_local(h, k, n) for n in range(3))
    H = H + sum(V_bond(h, k, i, j) for (i, j) in [(0, 1), (1, 2), (0, 2)])
    return H

print("Setup complete.")

Setup complete.


## 1. The Hamiltonian and the ground state

We work at $(h,k)=(1,3)$ as a concrete example (Section 5 repeats the test over many couplings).

In [2]:
h, k = 1, 3
H = H_tot(h, k)
g = ground_state(h, k)
print(f"M = {M_(h,k):.6f}   K = {K_(h,k):.6f}   c_H, c_V = {constants(h,k)}")
print("\nGround state |g> (amplitudes in the basis |q2 q1 q0>):")
labels = [format(i, '03b') for i in range(8)]
for lbl, amp in zip(labels, g):
    if abs(amp) > 1e-12:
        print(f"   |{lbl}> : {amp:+.6f}")
print("\n<g|g> =", np.vdot(g, g).real)

M = 0.245678   K = 3.605551   c_H, c_V = (np.float64(0.7955834968543577), np.float64(0.9414506867883019))

Ground state |g> (amplitudes in the basis |q2 q1 q0>):
   |001> : -0.226062
   |010> : -0.226062
   |100> : -0.226062
   |111> : +0.920156

<g|g> = 1.0000000000000004


## 2. $|g\rangle$ is the null space of $H_{\mathrm{tot}}$

Three equivalent demonstrations:

* **(a)** the residual vector $H_{\mathrm{tot}}\,|g\rangle$ is the zero vector;
* **(b)** the numerically computed null space (kernel) of $H_{\mathrm{tot}}$ is one-dimensional and coincides with $|g\rangle$;
* **(c)** the eigenvalue spectrum has a single zero (the ground state) and all other eigenvalues are strictly positive.

In [3]:
# (a) H|g> = 0 vector
res = H @ g
print("(a)  H_tot |g>  =", np.round(res, 12))
print("     || H_tot |g> ||  =", np.linalg.norm(res))

# (b) null space (kernel) of H_tot
from scipy.linalg import null_space
ns = null_space(H, rcond=1e-9)
print(f"\n(b)  dim(null space of H_tot) = {ns.shape[1]}")
print(f"     |<null_vector | g>|       = {abs(ns[:,0] @ g):.10f}   (=1 -> kernel is exactly span{{|g>}})")

# (c) spectrum
w = np.linalg.eigvalsh(H)
print("\n(c)  eigenvalues of H_tot:", np.round(w, 6))
print(f"     lowest eigenvalue   = {w[0]:+.2e}   (the null space)")
print(f"     smallest excited gap = {w[1]:+.4f}   (all other eigenvalues > 0)")

(a)  H_tot |g>  = [ 0.  0.  0.  0.  0.  0.  0. -0.]
     || H_tot |g> ||  = 2.220446049250313e-16



(b)  dim(null space of H_tot) = 1
     |<null_vector | g>|       = 1.0000000000   (=1 -> kernel is exactly span{|g>})

(c)  eigenvalues of H_tot: [ 0.      1.2111  1.2111  3.2111  3.2111  3.9196 14.4222 14.5026]
     lowest eigenvalue   = +6.27e-16   (the null space)
     smallest excited gap = +1.2111   (all other eigenvalues > 0)


## 3. Zero expectation value for every Hamiltonian part

The defining QET condition: the ground state carries zero mean energy for **each** local field,
**each** interaction bond, and the total Hamiltonian.

In [4]:
def expval(O): return float(g @ O @ g)

print("Local fields H_n = h Z_n + c_H :")
for n in range(3):
    print(f"   <g| H_{n} |g> = {expval(H_local(h,k,n)):+.3e}")

print("\nInteraction bonds V_ij = k X_i X_j + c_V :")
for (i, j) in [(0, 1), (1, 2), (0, 2)]:
    print(f"   <g| V_{i}{j} |g> = {expval(V_bond(h,k,i,j)):+.3e}")

print("\nGlobal:")
print(f"   <g| H_tot |g> = {expval(H):+.3e}")

allvals = ([expval(H_local(h,k,n)) for n in range(3)]
           + [expval(V_bond(h,k,i,j)) for (i,j) in [(0,1),(1,2),(0,2)]]
           + [expval(H)])
print(f"\nmax |expectation| over all 7 parts = {max(abs(v) for v in allvals):.2e}  -> all zero")

Local fields H_n = h Z_n + c_H :
   <g| H_0 |g> = +8.327e-17
   <g| H_1 |g> = +8.327e-17
   <g| H_2 |g> = +8.327e-17

Interaction bonds V_ij = k X_i X_j + c_V :
   <g| V_01 |g> = -2.776e-17
   <g| V_12 |g> = -5.551e-17
   <g| V_02 |g> = -5.551e-17

Global:
   <g| H_tot |g> = -2.043e-16

max |expectation| over all 7 parts = 2.04e-16  -> all zero


## 4. $|g\rangle$ is *not* an eigenstate of the individual parts

This is the non-trivial heart of QET: although the *expectation* of every local/semi-local part
vanishes, $|g\rangle$ is **not** an eigenstate of any individual $H_n$ or $V_{i,j}$. We show this
via the energy variance $\mathrm{Var}(O)=\langle g|O^2|g\rangle-\langle g|O|g\rangle^2$, which is
strictly positive for each part (a zero variance would mean $|g\rangle$ is an eigenstate). By
contrast, the variance of the *total* Hamiltonian is zero — $|g\rangle$ is a genuine (zero-energy)
eigenstate of $H_{\mathrm{tot}}$ only.

In [5]:
def variance(O): return float(g @ (O @ O) @ g) - float(g @ O @ g)**2

print("Energy variance of |g>  (>0  =>  NOT an eigenstate of that part):")
for n in range(3):
    print(f"   Var(H_{n})  = {variance(H_local(h,k,n)):.4f}")
for (i, j) in [(0, 1), (1, 2), (0, 2)]:
    print(f"   Var(V_{i}{j}) = {variance(V_bond(h,k,i,j)):.4f}")
print(f"\n   Var(H_tot) = {variance(H):.2e}   (~0  =>  |g> IS the zero-energy eigenstate of H_tot)")

Energy variance of |g>  (>0  =>  NOT an eigenstate of that part):
   Var(H_0)  = 0.3670
   Var(H_1)  = 0.3670
   Var(H_2)  = 0.3670
   Var(V_01) = 8.1137
   Var(V_12) = 8.1137
   Var(V_02) = 8.1137

   Var(H_tot) = 2.47e-15   (~0  =>  |g> IS the zero-energy eigenstate of H_tot)


## 5. The same holds for every admissible coupling $(h,k)$

We sweep several couplings and confirm, for each, that (i) $H_{\mathrm{tot}}|g\rangle=0$, (ii) the
null space is one-dimensional, and (iii) all seven expectation values vanish.

In [6]:
from scipy.linalg import null_space
hdr = f"{'(h,k)':>10} | {'||H|g>||':>10} | {'null dim':>8} | {'max|<H_n>|':>11} | {'max|<V_ij>|':>11} | {'<H_tot>':>10}"
print(hdr); print("-"*len(hdr))
for (h, k) in [(1, 2), (1, 3), (1, 4), (1.5, 1.0), (2, 8), (0.7, 2.3), (3, 10)]:
    H = H_tot(h, k); g = ground_state(h, k)
    locs = [abs(g @ H_local(h, k, n) @ g) for n in range(3)]
    sems = [abs(g @ V_bond(h, k, i, j) @ g) for (i, j) in [(0, 1), (1, 2), (0, 2)]]
    nd = null_space(H, rcond=1e-9).shape[1]
    print(f"{str((h,k)):>10} | {np.linalg.norm(H@g):10.2e} | {nd:8d} | "
          f"{max(locs):11.2e} | {max(sems):11.2e} | {float(g@H@g):+.2e}")
print("\nAll couplings: H_tot|g>=0, 1-D null space, and all expectations vanish.")

     (h,k) |   ||H|g>|| | null dim |  max|<H_n>| | max|<V_ij>| |    <H_tot>
---------------------------------------------------------------------------
    (1, 2) |   5.87e-16 |        1 |    2.78e-17 |    1.11e-16 | -5.50e-16
    (1, 3) |   2.22e-16 |        1 |    8.33e-17 |    5.55e-17 | -2.04e-16
    (1, 4) |   1.02e-15 |        1 |    2.78e-17 |    2.22e-16 | -9.26e-16
(1.5, 1.0) |   5.53e-16 |        1 |    1.53e-16 |    4.16e-17 | +5.49e-16
    (2, 8) |   2.04e-15 |        1 |    5.55e-17 |    4.44e-16 | -1.85e-15
(0.7, 2.3) |   3.14e-16 |        1 |    2.78e-17 |    2.78e-17 | -2.55e-16
   (3, 10) |   8.88e-16 |        1 |    3.33e-16 |    2.22e-16 | -8.14e-16

All couplings: H_tot|g>=0, 1-D null space, and all expectations vanish.


## Conclusion

$|g\rangle$ is the one-dimensional null space (zero-energy kernel) of $H_{\mathrm{tot}}$, and it
satisfies every zero-mean-energy condition — for each local field $H_n$, each interaction bond
$V_{i,j}$, and the total Hamiltonian — while being an eigenstate of *none* of the individual parts.
This is precisely the resource structure required by the QET protocol.